In [ ]:
!pip install openpyxl

In [ ]:
import s3fs
import rasterio
import xarray as xr
import rioxarray
import duckdb
import numpy as np
import matplotlib.pyplot as plt
import boto3
import pandas as pd
import geopandas as gpd
import sqlite3
from dask.distributed import Client, LocalCluster
from dask.diagnostics import ProgressBar

from const import host_spcodes

## Crosswalk TRY hydrology with TreeMap

In [ ]:
# Downlaod the treemap db to temporary directory
fs = s3fs.S3FileSystem()
fs.download(
    "nasa-cryo-scratch/s-kganz/treemap/Data/TreeMap2016_tree_table.db",
    "/tmp/TreeMap2016_tree_table.db"
)

# Connect
db = duckdb.connect("/tmp/TreeMap2016_tree_table.db")

In [ ]:
# Read the median trait file
traits = pd.read_excel("../data_in/tree_traits/GlobalTrees_Traits_Median.xlsx", sheet_name="GlobalTraits_Median")
traits = traits.set_index("spec.name")
traits.head()

In [ ]:
# Figure out what scientific names are in treemap and verify that these are all present
# in the traits database.
from functools import reduce

all_spcodes = reduce(set.union, map(set, host_spcodes.values()))
print(f"Found {len(all_spcodes)} species")

all_spec_names = db.sql(f'''
    SELECT DISTINCT SCIENTIFIC_NAME
    FROM TreeMap2016_tree_table
    WHERE SPCD IN {tuple(all_spcodes)}
''').df()
all_spec_names = all_spec_names.set_index("SCIENTIFIC_NAME")

In [ ]:
all_spec_names.iloc[~all_spec_names.index.isin(traits.index)]

In [ ]:
# Recode spp that failed to match
all_spec_names = all_spec_names.rename({
    "Pinus washoensis": "Pinus ponderosa",
    "Abies lasiocarpa var. arizonica": "Abies lasiocarpa"
})

# Assert that everything matches now
assert all_spec_names.index.isin(traits.index).all()

In [ ]:
# Subset the traits table. Ignore height bc we get that from FIA
trait_cols = ["gsmax", "P50", "rdmax", "WUE"]
traits_subset = traits.loc[all_spec_names.index][trait_cols]

In [ ]:
# Insert this table into the sqlite database so duckdb can use it
con = sqlite3.connect("/tmp/TreeMap2016_tree_table.db")
traits_subset.to_sql("Treemap_Median_Traits", con)

In [ ]:
# Verify that it worked
db.sql("SHOW TABLES")

In [ ]:
# Calculate the tree-density-weighted average of height and traits. Note this excludes
# all the non beetle-host species.
hydro_summary = db.sql(f'''
    SELECT
        tm_id,
        SUM(gsmax * TPA_UNADJ) / SUM(TPA_UNADJ) AS gsmax,
        SUM(P50 * TPA_UNADJ) / SUM(TPA_UNADJ) AS P50,
        SUM(rdmax * TPA_UNADJ) / SUM(TPA_UNADJ) AS rdmax,
        SUM(WUE * TPA_UNADJ) / SUM(TPA_UNADJ) AS WUE,
        SUM(HT * TPA_UNADJ) / SUM(TPA_UNADJ) as HT
    FROM
    (
        TreeMap2016_tree_table
        INNER JOIN Treemap_Median_Traits
        ON TreeMap2016_tree_table.SCIENTIFIC_NAME = Treemap_Median_Traits.SCIENTIFIC_NAME
    )
    GROUP BY tm_id
''').df().set_index("tm_id")

In [ ]:
# Join this onto a table with all tm_ids so we are guaranteed to always hit with .loc
# Many of these rows are all NaN because those TM IDs do not have a species that
# acts as insect host.
all_tmids = db.sql('''
    SELECT DISTINCT tm_id
    FROM TreeMap2016_tree_table
''').df().set_index("tm_id")

hydro_by_tmid = all_tmids.join(hydro_summary, how="left")
hydro_by_tmid.loc[2147483647] = np.nan

## Match median traits to TreeMap IDs

First open treemap and define the processing extent

In [ ]:
session = rasterio.session.AWSSession(boto3.Session(), requester_pays=True)

# Chunk sizes need to be a multiple of 30
chunks = dict(x=2880, y=2880)
treemap = rioxarray.open_rasterio(
    "s3://nasa-cryo-scratch/s-kganz/treemap/Data/TreeMap2016.tif", 
    band_as_variable=True,
    chunks=chunks
)
print(treemap.rio.crs)
treemap

In [ ]:
# Figure out processing extent
usfs_regions = gpd.read_file("../data_in/usfs_region_boundaries/usfs_regions_simple.shp").to_crs(treemap.rio.crs)
usfs_regions_explode = usfs_regions.geometry.explode()
usfs_regions_explode = usfs_regions_explode[usfs_regions_explode.geometry.area > 2e11]
bounds = usfs_regions_explode.total_bounds
xmin, ymin, xmax, ymax = bounds
print(bounds)

In [ ]:
# sel() must be exactly on the beginning/end of a chunk for ease of use
# with map_blocks(). So snap to the nearest coordinate and then snap
# to the nearest chunk
x_snap = treemap.x.sel(x=[xmin, xmax], method="nearest")
y_snap = treemap.y.sel(y=[ymin, ymax], method="nearest")
x_idx = np.where(treemap.x.isin(x_snap))[0]
y_idx = np.where(treemap.y.isin(y_snap))[0]

print("Before snapping:", x_idx, y_idx)

x_idx[0] = int(chunks["x"] * (np.floor(x_idx[0] / chunks["x"])))
x_idx[1] = int(chunks["x"] * (np.ceil(x_idx[1] / chunks["x"])))
y_idx[0] = int(chunks["y"] * (np.floor(y_idx[0] / chunks["y"])))
y_idx[1] = int(chunks["y"] * (np.ceil(y_idx[1] / chunks["y"])))

print("After snapping:", x_idx, y_idx)

In [ ]:
treemap_clip = treemap.isel(
    x=slice(*x_idx),
    y=slice(*y_idx) 
)
treemap_clip

In [ ]:
# Assert that all chunks are the same size.
for dim in treemap_clip.chunksizes:
    sizes = np.array(treemap_clip.chunksizes[dim])
    assert (sizes == sizes[0]).all()

In [ ]:
coarsen_factor = 10

def process_block(block: xr.Dataset) -> xr.DataArray:
    block_tmids = block.band_1.data.flatten()
    block_ba = xr.DataArray(
        data=hydro_by_tmid.loc[block_tmids].to_numpy().reshape(block.band_1.shape + (hydro_by_tmid.shape[1],)),
        dims=block.band_1.dims + ("hydro_trait",),
        coords=dict(hydro_trait=hydro_by_tmid.columns, **block.coords)
    )
    block_coarse = block_ba.coarsen(dict(y=coarsen_factor, x=coarsen_factor), boundary="trim").mean()
    return block_coarse

In [ ]:
template = treemap_clip.band_1.coarsen(x=coarsen_factor, y=coarsen_factor, boundary="trim").mean()
template = template.expand_dims(hydro_trait=hydro_by_tmid.columns, axis=-1)
template

In [ ]:
trait_da = xr.map_blocks(
    func=process_block,
    obj=treemap_clip,
    template=template
)

In [ ]:
with ProgressBar():
    trait_da = trait_da.compute()

In [ ]:
trait_da

In [ ]:
hydro_coarse = trait_da.coarsen(x=10, y=10, boundary="pad").mean()

In [ ]:
from cartopy import crs as ccrs
from cartopy import feature as cfeature

source_proj = ccrs.AlbersEqualArea(
    central_latitude=23,
    central_longitude=-96,
    standard_parallels=(29.5, 45.5)
)
target_proj = ccrs.Mercator()

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(12, 6), subplot_kw=dict(projection=target_proj))
xmin, ymin, xmax, ymax = hydro_coarse.rio.transform_bounds(3857)

for hydro_trait, ax in zip(hydro_by_tmid.columns, axes.flat[:5]):
    hydro_coarse.sel(hydro_trait=hydro_trait).plot(ax=ax, transform=source_proj, add_colorbar=True, add_labels=False, xlim=[xmin, xmax], ylim=[ymin, ymax])
    ax.set_title(hydro_trait)
    ax.coastlines()
    ax.add_feature(cfeature.STATES)

axes.flat[-1].axis("off")
plt.show()